In [1]:
import torch
import numpy as np
import math
from torch.utils.data import Dataset, DataLoader
from torch.autograd import Variable
from sklearn.model_selection import train_test_split
import torch.nn as nn
import matplotlib.pyplot as plt
from sklearn import preprocessing
from sklearn.metrics import r2_score
import random
import matplotlib as mpl
import os
import gc
import pandas as pd
import csv
from numpy import *

from torch.utils.tensorboard import SummaryWriter
from datetime import date
from sax import sax_tokenizer
# from generate_property import output_property

In [ ]:
data_dir = '../generating_raw_data/data_small/'
total_files = 10000
max_len = 500
num_f = 6

input_x = np.zeros((total_files,max_len,num_f))
output_y = np.zeros((total_files,))

for i in range(total_files):
    temp_x = np.load(f'{data_dir}sample_{i}.npy', allow_pickle=True)
    temp_y = np.load(f'{data_dir}target_{i}.npy', allow_pickle=True)
       
    input_x[i,...] = temp_x
    output_y[i,...] = temp_y


print('Size of input_x', input_x.shape)
print('Size of output_y', output_y.shape)

# input_x = input_x[0:100,...]
# output_y = output_y[0:100,]

Size of input_x (10000, 500, 6)
Size of output_y (10000,)


**SAX: Symbolic Representation**

In [3]:
# '''Write a function to map the sax representation to the actual sequence length'''
# def decode(sax_rep, original_len,word_len):
#     decode_seq = np.zeros((sax_rep.shape[0], original_len))
#     l = sax_rep.shape[0]
#     count = 0
#     while count < l:
#         print('use this', sax_rep[0:3,count])
#         decode_seq[:,count:count+word_len] = sax_rep[:,count]
#         print('Result', decode_seq[0:3,count:count+word_len])
#         count += word_len
#         print(aaa)
    
#     return decode_seq

In [4]:
category = 2
np.save('./num_category', category)
word_len = 1
x_sax = np.zeros(input_x.shape)

for i in range(len(x_sax)):
    start = 0
    for j in range(input_x.shape[-1]):
        temp = sax_tokenizer(input_x[i,:,j].tolist(),alphabet_size=category, word_length=word_len) #+ start
        x_sax[i,:,j] =  np.array(temp) + start
        # print(temp)
        start += category
    if i%500 == 0:
        print(f'Done with {i}')


# x_sax_decoded = decode(x_sax, input_x.shape[1],word_len)
        

Done with 0
Done with 500
Done with 1000
Done with 1500
Done with 2000
Done with 2500
Done with 3000
Done with 3500
Done with 4000
Done with 4500
Done with 5000
Done with 5500
Done with 6000
Done with 6500
Done with 7000
Done with 7500
Done with 8000
Done with 8500
Done with 9000
Done with 9500


In [5]:
x_sax = x_sax.astype(int)
x_sax_ohe = np.zeros((input_x.shape[0], input_x.shape[1], category*input_x.shape[-1]))

for i in range(len(x_sax_ohe)):
    for j in range(x_sax_ohe.shape[1]): 
        idx = x_sax[i,j,:].tolist()
        x_sax_ohe[i,j,idx] = 1

In [6]:
print(x_sax[50,50,:])
print(x_sax_ohe[50,50,:])

[ 1  2  5  7  9 11]
[0. 1. 1. 0. 0. 1. 0. 1. 0. 1. 0. 1.]


In [7]:
seq_length =np.ones(output_y.shape)*max_len

In [8]:
seed = 50 ## [10,50,70]
all_ex = np.arange(input_x.shape[0])
X, x_test, _, _ = train_test_split( all_ex, all_ex, test_size=0.05,random_state=seed) ## [10,50,70]
x_train, x_valid, _, _ = train_test_split( X, X, test_size=0.05263,random_state=seed)
_, x_train, _, _ = train_test_split( x_train, x_train, test_size=0.55555,random_state=50)

print('Train',x_train.shape)
print('Test' ,x_test.shape)
print('Valid',x_valid.shape)

print(input_x[x_train].shape, seq_length[x_train].shape, output_y[x_train].shape)

print(np.sum(output_y[x_train]),np.sum(output_y[x_valid]),np.sum(output_y[x_test]))

Train (5000,)
Test (500,)
Valid (500,)
(5000, 500, 6) (5000,) (5000,)
2503.0 242.0 244.0


In [9]:
np.save('./x_train', input_x[x_train])
np.save(f'./sax_train_{category}', x_sax_ohe[x_train])
np.save('./len_train', seq_length[x_train])
np.save('./y_train', output_y[x_train])

np.save('./x_valid', input_x[x_valid])
np.save(f'./sax_valid_{category}', x_sax_ohe[x_valid])
np.save('./len_valid', seq_length[x_valid])
np.save('./y_valid', output_y[x_valid])

np.save('./x_test', input_x[x_test])
np.save(f'./sax_test_{category}', x_sax_ohe[x_test])
np.save('./len_test', seq_length[x_test])
np.save('./y_test', output_y[x_test])
np.save('./test_idx', x_test)

In [10]:
print(x_test)

[9102 7868 4176 4161 8770 8919 3884 1590  617 1562 1043 3494 2884 1505
  102 1912 7294 2464 7070 6468 6651 5498 8821 3777 4913 3614 6770 9912
 7044 6637    4  944 7503 2069 1441 8933 2354 4995 9612 9880 6529 3534
 1072 1261 8070 7138 5859 5919 5802  158 7666 5540 8291 2216 9625 9801
  329 7729 1924  434 8470 3396 2089 9736 5522 3437 2027  801 3410 6251
 5706 1249 1218 9428 2095 4020 7187 8219 9682 5430 2949  846 9408 1433
 3761  708 7879 5316 7389 8898 7533  428 6367  829 3703 5238 9353 6264
 5809 6423   12 2319 6944 5699 7940 6133 4639 1888 7493 2674 2261 8825
  956 1489 8718 3737 4162 4244 3188 8421 1980 8371 8022  438 3471 6334
 6487  513 2415 8883 1514 2567 7264 8942 7753 9051 1615 2229 6299 8255
 3873 1681 3416 7481 6149  842 5019 4622 1296  914 9500 2937 9918 5096
 5778 2628 1832 3201  203 1345  253 2413  542 1823 1944 6961 8625 7574
 4079 7751 6575 6048 8049 6747 9897 9182 3200 3426 5380 8202  453  864
  246 4609 9426 1226  976    2 8564 7926 8335 4334 3731 3715 4364 5094
   61 

In [11]:
x_sax_ohe[x_test].shape

(500, 500, 12)